# HW4 A: Probing HuBERT (15 automatic + 9 report points; up to 10 bonus)
Train small probes to measure which information is accessible in HuBERT's hidden
states. Compare all 13 outputs: the input to the Transformer stack (0, after convolutional features,
projection, and positional processing) and the 12 Transformer layers (1–12). Freeze HuBERT; train a separate probe per layer.
Use the same architecture and training budget for a fair comparison. Probe accuracy
measures information accessible to this probe, not all information in a layer.

**Grading:** 12 points for measured probe performance (6 phoneme, 6 MFCC),
3 for implementation, and 9 for the report. Each of the six code checks is worth
0.5 points: probe, standardization, classification training, regression training,
valid-frame pooling, and speaker extraction. The shared probe/standardization
points are split equally between A1/A2: each has 1 code + 6 performance + 3 report
points. A3 has 1 code + 3 report points; it is a visualization task with no trained
speaker classifier. The personal-voice experiment remains up to 10 bonus points.
The compact data uses LibriSpeech with automatic MFA phone alignments, which can
contain errors. Performance is recomputed from predictions, not the numbers in plots.

**Written answers:** submit one searchable, text-selectable `report.pdf` using the
five section IDs **A1, A2, A3, B1, C1** in `report_template.txt`. Put conceptual
answers and requested figures in that PDF; notebook markdown answers are not
manually graded. Aim for 3–5 sentences per section, plus the requested figures or
token sequence. There is no minimum page count. No extra experiments are required
for B1/C1. Mark unfinished work `Not completed` and explain what is available.
The shared report is worth 15 points (A: 9, B: 3, C: 3); the autograder is worth 85.


## Setup
Open this notebook on Colab and select a GPU. Setup automatically downloads the
small support bundle from the public course Hugging Face repository; no account,
token, or TA Drive access is needed. The next cell downloads only this part's data.
Downloads use a fixed release and verify checksums. Setup installs Python
dependencies while preserving Colab's matched PyTorch/CUDA installation.
Run setup once in a fresh runtime. If pip asks for a restart, restart and rerun setup.

Implement each TODO before running the cells that use it. For a quick check of
**your implementation**, set `SMOKE_TEST = True`. Smoke results are not completed experiments.
Definitions used by the autograder must be top-level functions/classes with the
specified names and signatures. The grader provides the standard imports shown
in setup; put any additional imports inside your functions. Keep helper constants
inside functions or as class attributes: top-level assignments are not executed.
A syntax error skips that cell; other independently implemented questions still count.

### What you must complete

| Task | Implement | Run with `SMOKE_TEST=False` | Saved work |
|---|---|---|---|
| A1 | `Probe`, `standardize_features`, `train_phoneme_probe` | All 13 layers, 5 epochs each | Per-layer phone predictions and accuracy plot |
| A2 | `train_mfcc_probe`, reusing the same probe and preprocessing | All 13 layers, 50 epochs each | Per-layer MFCC predictions and R² plot |
| A3 | `mean_pool`, `extract_speaker_embedding` | Frozen HuBERT on the same 1,000 recordings, all 13 layers | Speaker t-SNE panels |

Data loading, experiment loops, plotting, prediction-file writing, and ZIP building
are supplied. You write the two probe training loops, including validation-based
checkpoint selection. Use cross entropy for phones and MSE for MFCCs. Never fit
preprocessing or choose checkpoints using test labels. Return the selected model's
predictions in the loader's original test-row order; do not shuffle test outputs.
In normal mode there are **1,893 test frames**: phone predictions have shape `[1893]`
and contain integer class IDs; MFCC predictions have shape `[1893,13]` in the
original target units. `test_predictions` may be a list, NumPy array, or CPU tensor.
The experiment cells save each finished layer immediately. If a later layer fails,
submit the files already saved and fix the remaining code when you can.

**Debug before the full run:** first run one layer in smoke mode. Check that losses
are finite, phone accuracy is in `[0,1]`, and MFCC R² is finite (negative is valid).
Afterwards set smoke mode to `False` and rerun the experiment cells. Use a new
`RESULTS` directory when changing experiment settings so saved layers are from
one consistent run. In A3, a single recording must give `[13,768]` when
`layer_idx=None`; plotting and extraction are supplied around your implementation.


In [ ]:
REQUIREMENTS = 'requirements.txt'
# Colab: select Runtime > Change runtime type > GPU before training.
import sys, subprocess
from pathlib import Path

# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v2'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v2', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = False  # True: only a few steps, for debugging your implementation.
DATA_ROOT = Path(os.environ.get('HW4_DATA_ROOT', 'data'))
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
# Checkpoints/results disappear when Colab deletes a runtime. Copy them to Drive
# or download them before disconnecting. Submission ZIPs contain no checkpoints.

In [ ]:
# Download only Part A data; verified cached copies are reused.
from course_download import ensure_course_data
DATA_ARCHIVE = None  # Optional offline override: Path("hw4-data-a.zip")
ensure_course_data('a', data_root=DATA_ROOT, archive=DATA_ARCHIVE)


In [ ]:
import json, math, copy
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, unpad_sequence
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt
from hw4_runtime import load_phoneme_data, load_mfcc_data, load_audio
np.random.seed(183); torch.manual_seed(183)
device = DEVICE

## Task 1: Phoneme classification
The precomputed tensors have shape `[frames, 768]`; label files contain aligned
phone names. All layers use the same frames and split. Fit preprocessing on the
training split only. Use development loss to choose checkpoints and evaluate the
test split only after choosing training settings.

Implement `Probe` with `Linear(input_dim, hidden_dim)`, ReLU, and
`Linear(hidden_dim, output_dim)`; use hidden size 128 in the experiments. Return
raw logits for classification and unrestricted real values for regression.
### Performance rubric for A1/A2 (6 points each)

Every layer contributes an equal **6/13 points** per task. The table gives the
full-credit target for that layer. Accuracy or R² at/above that target earns its
full share. Between the baseline and target, credit increases linearly; at/below
the baseline, credit is zero. Phone baseline is **137/1893 = 7.2372%**, obtained
by always predicting the most common training phone. MFCC baseline is **R² = 0**.
For either task, the layer's fraction is
`clip((metric - baseline) / (target - baseline), 0, 1)`.
Missing or invalid layer predictions earn zero only for that layer; the denominator
stays 13. Each submitted layer must contain all test frames in the original order.

| HuBERT layer | Full-credit phone accuracy | Full-credit MFCC R² |
|---|---:|---:|
| 0 | 35% | 0.60 |
| 1 | 40% | 0.45 |
| 2 | 45% | 0.40 |
| 3 | 50% | 0.35 |
| 4 | 55% | 0.25 |
| 5 | 60% | 0.25 |
| 6 | 65% | 0.20 |
| 7 | 65% | 0.20 |
| 8 | 65% | 0.20 |
| 9 | 65% | 0.20 |
| 10 | 65% | 0.25 |
| 11 | 65% | 0.25 |
| 12 | 60% | 0.25 |

The targets leave room below the staff results at the supplied training budgets.
We score layers separately because acoustic/phonetic information varies by layer;
there is no requirement that every layer perform like the best layer.
The supplied export cells write the predictions for you. Do not submit labels as
predictions or edit model outputs by hand. Report interpretation is graded
independently of these performance thresholds.


In [ ]:
base_folder = DATA_ROOT / 'probes'
train_features, train_labels, dev_features, dev_labels, test_features, test_labels = load_phoneme_data(base_folder, 0)
print('Train:', train_features.shape, 'Dev:', dev_features.shape, 'Test:', test_features.shape)
print('Phone classes:', int(torch.cat([train_labels, dev_labels, test_labels]).max()) + 1)

In [ ]:
class Probe(nn.Module):

    def __init__(self, input_dim=768, hidden_dim=128, output_dim=50):
        """Call super().__init__(); construct exactly Linear -> ReLU -> Linear using the given dimensions."""
        raise NotImplementedError('TODO: __init__')

    def forward(self, x):
        """Return self.network(x)."""
        raise NotImplementedError('TODO: forward')

In [ ]:
def standardize_features(train, dev, test, eps=1e-05):
    """Return (train_z, dev_z, test_z). Compute mean and POPULATION standard deviation (unbiased=False) per feature on train only. Clamp std to at least eps. Do not modify input tensors."""
    raise NotImplementedError('TODO: standardize_features')

In [ ]:
def train_phoneme_probe(base_folder, layer_idx, num_epochs=10, batch_size=512, learning_rate=0.001):
    """Load the six tensors using load_phoneme_data(base_folder, layer_idx). Use standardize_features and Probe(hidden_dim=128), cross entropy, Adam, minibatches, and the supplied learning rate. Keep datasets on CPU and move batches to device. Choose a checkpoint by development loss, then evaluate test once. Return {"layer": layer_idx, "test_acc": float in [0,1], "test_predictions": predicted class IDs [N_test] in original test order}. Use max label + 1 as output size, not the number of distinct labels in one split. Honor num_epochs and batch_size."""
    raise NotImplementedError('TODO: train_phoneme_probe')

In [ ]:
from probe_submission import save_probe_predictions
all_results = []
for layer_idx in ([0] if SMOKE_TEST else range(13)):
    result = train_phoneme_probe(base_folder, layer_idx, num_epochs=1 if SMOKE_TEST else 5, batch_size=256)
    save_probe_predictions('phoneme', result, RESULTS / 'a')
    result = {key: value for key, value in result.items() if key != 'test_predictions'}
    all_results.append(result)
    print(result)
    (RESULTS / 'a').mkdir(exist_ok=True)
    (RESULTS / 'a/phoneme_results.json').write_text(json.dumps(all_results, indent=2))
fig, ax = plt.subplots()
ax.plot([r['layer'] for r in all_results], [r['test_acc'] for r in all_results], marker='o')
ax.set(xlabel='HuBERT layer', ylabel='Test accuracy', xticks=range(13))
fig.tight_layout(); fig.savefig(RESULTS / 'a/phoneme_accuracy.png', dpi=150)
plt.show()

### Report A1: Phoneme probing (3 manual points)
Put this answer in the **A1** section of `report.pdf`.

- **Figure A1 (1 point):** accuracy versus all 13 HuBERT layers, with labeled axes.
  Use your measured results; label any unfinished layers or smoke-only results.
- **Answer A1 (2 points):** identify the best observed layer, explain what the trend
  suggests about phonetic information accessible to the probe, and give one
  limitation of the probe or automatic phone alignments. Use 3–5 sentences.

List your training budget once in the report's experiment summary. No particular
accuracy is required for report credit.


## Task 2: MFCC regression
Predict 13 MFCC coefficients from each layer using the same two-layer probe.
MFCCs summarize the spectral envelope of short audio windows. Targets are aligned
to HuBERT frame centers; the release metadata records the extraction parameters.
Use MSE for training and **uniform-average R²** across the 13 coefficients for
evaluation. R² may be negative. Standardize input features using training statistics.
The provided experiment uses 50 epochs per layer because unscaled MFCC targets
need more updates than the phoneme classifier. A short five-epoch check can still
have negative R². Use the same budget for every layer, select by development loss,
and report any smaller budget you used. Code and report credit remain available even with low R²; performance credit follows the rubric below.


In [ ]:
def train_mfcc_probe(base_folder, layer_idx, num_epochs=10, batch_size=512, learning_rate=0.001):
    """Load the six tensors with load_mfcc_data. Train Probe(input_dim, 128, output_dim=13) using standardize_features, MSE and Adam. Choose a checkpoint by development loss. Return {"layer": layer_idx, "overall_r2": float, "test_predictions": predicted MFCCs [N_test,13] in original test order}, using sklearn.metrics.r2_score(test_targets, predictions, multioutput="uniform_average"). Keep minibatches small and honor the arguments."""
    raise NotImplementedError('TODO: train_mfcc_probe')

In [ ]:
from probe_submission import save_probe_predictions
mfcc_all_results = []
for layer_idx in ([0] if SMOKE_TEST else range(13)):
    result = train_mfcc_probe(base_folder, layer_idx, num_epochs=1 if SMOKE_TEST else 50, batch_size=256)
    save_probe_predictions('mfcc', result, RESULTS / 'a')
    result = {key: value for key, value in result.items() if key != 'test_predictions'}
    mfcc_all_results.append(result)
    print(result)
    (RESULTS / 'a').mkdir(exist_ok=True)
    (RESULTS / 'a/mfcc_results.json').write_text(json.dumps(mfcc_all_results, indent=2))
fig, ax = plt.subplots()
ax.plot([r['layer'] for r in mfcc_all_results], [r['overall_r2'] for r in mfcc_all_results], marker='o')
ax.set(xlabel='HuBERT layer', ylabel='Test R²', xticks=range(13))
fig.tight_layout(); fig.savefig(RESULTS / 'a/mfcc_r2.png', dpi=150)
plt.show()

### Report A2: MFCC probing (3 manual points)
Put this answer in the **A2** section of `report.pdf`.

- **Figure A2 (1 point):** uniform-average test R² versus all 13 HuBERT layers,
  with labeled axes. Include negative values if observed.
- **Answer A2 (2 points):** identify the best observed layer, explain what layer 0
  represents, and compare the acoustic trend with the phoneme trend from A1.
  Use 3–5 sentences; lower scores do not by themselves lose interpretation credit.


## Task 3: Speaker embeddings
The next cell downloads the course speaker recordings from Hugging Face. Extract HuBERT hidden states in evaluation mode,
and mean-pool valid frames. Compare a consistent subset of 20 speakers across all
13 layers. Use t-SNE with a fixed random seed and perplexity smaller than the number
of recordings. Do not treat t-SNE distances between separate plots as calibrated.

In [ ]:
def mean_pool(hidden_states, lengths):
    """hidden_states is [B,T,D], lengths is [B] with positive valid-frame counts. Return [B,D], averaging only frames t < lengths[b]. Preserve gradients and device."""
    raise NotImplementedError('TODO: mean_pool')

In [ ]:
from transformers import HubertModel, Wav2Vec2FeatureExtractor
from sklearn.manifold import TSNE
import os
from hw4_runtime import load_hubert
hubert = load_hubert().to(device).eval()
# Original speaker dataset: 20 speakers, 50 recordings each (about 215 MiB).
from course_download import ensure_course_data
ensure_course_data("speakers", data_root=DATA_ROOT)
audio_files = sorted((DATA_ROOT / 'speakers').rglob('*.flac'))
if not audio_files:
    raise FileNotFoundError('Speaker download is incomplete; rerun this cell.')

In [ ]:
def extract_speaker_embedding(audio_path, model, feature_extractor=None, layer_idx=0):
    """Return [768] for one layer, or [13,768] when layer_idx=None.
    Use load_audio, normalize the mono 16 kHz waveform, model.eval(), and
    torch.no_grad(). Request output_hidden_states=True, then mean_pool.
    Returning every layer from one forward pass saves substantial compute.
    """
    raise NotImplementedError('TODO: extract_speaker_embedding')

# All layers are collected together. Speaker ID is the first filename component.
selected = audio_files[::max(1, len(audio_files) // 8)][:8] if SMOKE_TEST else audio_files
embeddings, speaker_ids = [], []
for path in selected:
    embeddings.append(extract_speaker_embedding(path, hubert, layer_idx=None))
    speaker_ids.append(Path(path).stem.split('-')[0])
embeddings = torch.stack(embeddings).cpu().numpy()
labels = np.array([sorted(set(speaker_ids)).index(s) for s in speaker_ids])
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for layer, ax in enumerate(axes.flat):
    if layer >= 13:
        ax.axis('off'); continue
    xy = TSNE(n_components=2, random_state=183, init='pca', learning_rate='auto',
              perplexity=min(30, len(selected)-1), max_iter=500).fit_transform(embeddings[:, layer])
    ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap='tab20', s=8)
    ax.set_title(f'Layer {layer}')
from matplotlib.lines import Line2D
speakers = sorted(set(speaker_ids))
handles = [Line2D([], [], marker='o', linestyle='', color=plt.get_cmap('tab20')(i / max(1, len(speakers)-1)), label=s)
           for i, s in enumerate(speakers)]
fig.legend(handles=handles, loc='lower center', ncol=10, title='Speaker ID')
fig.tight_layout(rect=(0, 0.05, 1, 1))
(RESULTS / 'a').mkdir(exist_ok=True)
fig.savefig(RESULTS / 'a/speakers.png', dpi=150)
plt.show()


### Report A3: Speaker analysis (3 manual points)
Put this answer in the **A3** section of `report.pdf`.

- **Figure A3 (1 point):** the 13 speaker t-SNE plots, preferably in one or more
  legible panels. Label the layer and speaker colors; give the seed/perplexity in
  the caption and use the same recordings across layers.
- **Answer A3 (2 points):** identify which layers separate speakers most clearly,
  relate this to A1/A2, and state one limitation of interpreting t-SNE. Use 3–5
  sentences. Distances in separately fitted t-SNE plots are not directly comparable.

### Optional bonus: your own voice (up to 10 manual points)
Use the **BONUS** section only if you attempted this original optional experiment.
Record five different sentences with each of **two microphones** in each of the
same **three environments**: 2 × 3 × 5 = **30 recordings**, in six condition groups.
Mean-pool your HuBERT embeddings and add all six groups to one of the speaker plots,
using a layer of your choice. Label microphone/environment groups clearly.

- Plot with the six added groups: 6 points.
- Explain their placement relative to the other speakers: 2 points.
- Explain their placement relative to each other: 2 points.

Submit the plot, recording conditions, layer choice, and discussion in the PDF;
do not include personal audio in the submission ZIP.


## Prepare your submission
**Default: submit one combined ZIP containing one shared `report.pdf`.**
The PDF covers **A1, A2, A3, B1, C1** (and BONUS if attempted). Use
`report_template.txt`, keep all five headings, and write `Not completed` only for
answers you did not finish. You may answer B1/C1 without training those models.
Export one PDF with selectable text from Google Docs, Word, or LaTeX. If you wrote
separate reports, combine them into this document before uploading; the helper
**does not merge PDFs**. Notebook PDFs are not required.

**Typical A → B → C workflow**

1. **After A:** run its final cell. Download `submission.zip` and rename it
   `hw4-a.zip` on your computer. At the report prompt, upload your current combined
   PDF or cancel if it is not ready.
2. **After B:** leave `include_previous=True` (B's default). At the **earlier ZIP**
   prompt, upload `hw4-a.zip`. At the separate **report** prompt, upload an updated
   combined PDF, or cancel to keep the report already in the ZIP. The cell adds
   your current B notebook and available B predictions. Save the download as
   `hw4-ab.zip`.
3. **After C:** leave `include_previous=True` (C's default). Upload `hw4-ab.zip`
   at the earlier-ZIP prompt, then your final combined `report.pdf` at the report
   prompt. The cell adds C and downloads the final `submission.zip`. Upload
   **that one ZIP** to the HW4 Programming Assignment on Gradescope.

**Stopping early or working in another order**

| Work you want graded | What to do |
|---|---|
| A + B only | Finish step 2 with your final report, then submit `hw4-ab.zip`. You do **not** need to open or run C. |
| A only, or part of A | Submit A's ZIP with the report answers you completed. |
| Only B or C | Cancel the earlier-ZIP prompt; the cell still packages the current notebook and available results. |
| Separate A and B ZIPs, finishing C | Select both A and B ZIPs together at C's earlier-ZIP prompt, then upload the shared report at the next prompt. |
| Another order, or updating earlier work | Set `include_previous=True` in whichever notebook you package last and import your latest combined ZIP. Current notebook edits and available local results replace their earlier copies. |

Use the **latest** ZIP for each part; do not select both an old and a new version
of the same part. Keep `include_report=True`: cancelling its upload preserves an
existing report; setting it to `False` deliberately leaves the report out.
Missing code, predictions, or a PDF do not block grading of other completed work.
Missing C does not reduce A/B credit. Save each ZIP before disconnecting Colab.

**Before uploading:** read the printed notebook summary, file list, and warnings.
Check that all notebooks you want graded and their available prediction files
are included. Open `report.pdf` inside the ZIP to confirm it is your latest report.
Warnings for intentionally unfinished questions are expected; fix warnings about
work you intended to include. Every Gradescope resubmission must contain **all**
work you want graded; earlier attempts are not combined with the newest one.

**Packaging after a runtime reset:** you can run just the final cell; it downloads
the small support bundle from Hugging Face only if its helper is missing. No GPU, dataset download,
dependency installation, or training is needed for packaging. Set
`include_previous=True` and import your saved submission ZIP to recover earlier
results. If newer results exist only on Drive, mount it and restore your `RESULTS`
path first; the helper cannot recover files from a deleted runtime.

A performance requires `results/a/phoneme_layer_0.json` through layer 12 and
`results/a/mfcc_layer_0.json` through layer 12, written from your returned
`test_predictions`. Available layers are enough for a partial submission;
summary metrics or plots alone do not earn performance points.

The ZIP contains notebooks, available small predictions/plots, and the shared PDF.
It excludes checkpoints, pretrained weights, datasets, and audio. Keep those
separately for resuming training; do not upload them or the support bundle to
Gradescope. Locally, save your edited notebooks, `results/`, and `report.pdf`
together, then run `python make_submission.py` in that folder.


In [ ]:
include_report = True  # Upload one combined PDF, or cancel to retain an earlier one.
include_previous = False  # Merge earlier submission ZIPs; cancel if none.

# Packaging needs no training dependencies, GPU, or data download.
# Course release settings are supplied by staff; no student login is needed.
COURSE_REPO_ID = 'Louis0324/eecs183-hw4'
COURSE_REVISION = 'hw4-2026-v2'
import os
os.environ.setdefault("HW4_HF_REPO", COURSE_REPO_ID)
os.environ.setdefault("HW4_HF_REVISION", COURSE_REVISION)

def ensure_hw4_support(repo_id=None, revision='hw4-2026-v2', packager_only=False):
    """Fetch the public support bundle, or use a local/uploaded offline copy."""
    import hashlib
    from io import BytesIO
    import json
    from pathlib import Path
    import re
    from urllib.parse import quote
    from urllib.request import urlopen
    import zipfile

    required = 'make_submission.py' if packager_only else 'course_download.py'
    if Path(required).is_file():
        return
    local_bundle = Path('hw4-student.zip')
    if local_bundle.is_file():
        content = local_bundle.read_bytes()
    elif repo_id:
        if not re.fullmatch(r'[\w.-]+/[\w.-]+', repo_id):
            raise ValueError('Expected a Hugging Face repository ID like COURSE_ORG/eecs183-hw4.')
        base = f'https://huggingface.co/datasets/{repo_id}/resolve/{quote(revision, safe="")}/'
        print('Downloading the small HW4 support bundle from Hugging Face; no login is required.')
        try:
            with urlopen(base + 'release_manifest.json', timeout=30) as response:
                manifest = json.loads(response.read(1024 * 1024))
            spec = manifest['files']['hw4-student.zip']
            if not 0 < spec['bytes'] < 30 * 1024 * 1024:
                raise ValueError('Invalid support bundle size in the release manifest.')
            with urlopen(base + 'hw4-student.zip', timeout=30) as response:
                content = response.read(spec['bytes'] + 1)
            if len(content) != spec['bytes'] or hashlib.sha256(content).hexdigest() != spec['sha256']:
                raise ValueError('Support bundle checksum mismatch; rerun to download a fresh copy.')
        except Exception as error:
            raise RuntimeError('Could not download the HW4 support release. Rerun this cell; '
                               'if needed, download hw4-student.zip from the course Hugging Face repo '
                               'and upload it through the Files sidebar before retrying.') from error
    else:
        # Allows local staff testing and offline/manual use before publication.
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError('Put hw4-student.zip in this folder or configure the course Hugging Face repository.') from error
        print('Upload hw4-student.zip to load the supplied helpers.')
        uploaded = files.upload()
        archives = [value for name, value in uploaded.items() if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('Upload exactly one hw4-student.zip support bundle, then rerun.')
        content = archives[0]

    with zipfile.ZipFile(BytesIO(content)) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt support ZIP; download a fresh copy.')
        for item in archive.infolist():
            path = Path(item.filename)
            if (path.is_absolute() or '..' in path.parts or
                    ((item.external_attr >> 16) & 0o170000) == 0o120000 or
                    not path.resolve().is_relative_to(Path.cwd().resolve())):
                raise ValueError('Unsafe support archive path.')
            # Browser/local notebook edits must never be replaced by starter copies.
            if item.is_dir() or path.suffix == '.ipynb' or (packager_only and item.filename != required):
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(item))
    if not Path(required).is_file():
        raise ValueError(f'This ZIP is missing {required}; use the current hw4-student.zip support bundle.')

ensure_hw4_support(COURSE_REPO_ID, COURSE_REVISION, packager_only=True)
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
from make_submission import build_submission, from_colab
if colab_files is not None:
    from_colab('a', include_report=include_report, include_previous=include_previous,
               results_dir=globals().get("RESULTS", "results"))
else:
    build_submission(report="report.pdf" if include_report else None)
